# Module 11: Fine-tune vs. RAG vs. prompt

Run the cells top to bottom. Cells marked **YOUR TURN** are the build: they run as they are, but they won't pass the done-when checks until you fill them in. All data in this kit is synthetic.

In [ ]:
# Runs only on Google Colab: fetch the starter kit and its packages, and copy any keys
# you saved in Colab's Secrets panel into the environment. Locally this does nothing.
import os, sys
if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/Go-Everyway/ai-track-starter /content/ai-track-starter
    %cd /content/ai-track-starter
    %pip install -q -r requirements.txt
    from google.colab import userdata
    for name in ["LLM_API_KEY", "LLM_PROVIDER", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL", "LLM_DAILY_BUDGET_USD"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set in Secrets; lib/llm.py uses its default

In [ ]:
# Find the repo root so paths work from notebooks/ or from Colab.
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "leads_clean.csv").exists())
DATA = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("starter kit at", ROOT)

## 1. The question

Module 7's outreach-angle step works. Before anyone suggests fine-tuning it, compare the
cheaper rungs of the ladder on the same test suite, with real token counts and your own
prices. This notebook compares three approaches:

- **A. Prompt only:** Module 7's system prompt with its worked example removed.
- **B. Prompt + one example:** Module 7's v3, as shipped.
- **C. Prompt + example + retrieved facts:** B, plus the three Tallyhawk passages Module 8's
  keyword search finds for each company.

In mock mode (no key), costs come from real token counts but quality is plumbing only.

In [ ]:
import json
import time

import tiktoken

from lib import llm, outreach, rag
from evals.run import run_all
from evals.registry import mock_mode

enc = tiktoken.get_encoding("o200k_base")
tokens = lambda text: len(enc.encode(text))
print("mode:", "MOCK (costs from real token counts; quality is plumbing only)" if mock_mode() else "real model")

## 2. Build the three system prompts

In [ ]:
SYSTEM_B = outreach.SYSTEM_V2
SYSTEM_A = SYSTEM_B.split("\n\nExample\n")[0]  # same rules, no worked example

kb = rag.build_index(rag.load_docs(DATA / "docs"), embed=False, size=40, overlap=10, prefix_title=True)

def system_c(company):
    hits = rag.search(kb, company["description"], k=3, mode="keyword")
    facts = "\n".join(f"- {h.chunk.text}" for h in hits)
    return SYSTEM_B + "\n\nFacts about Tallyhawk you may use in the opener (and nothing else about Tallyhawk):\n" + facts

APPROACHES = {
    "A: prompt only": lambda company: SYSTEM_A,
    "B: prompt + 1 example": lambda company: SYSTEM_B,
    "C: + retrieved facts": system_c,
}
for name, build in APPROACHES.items():
    print(f"{name:<24} system prompt for one company: {tokens(build({'description': 'wholesale plumbing parts distributor'})):>5} tokens")

## 3. Run every approach on the same suite

`run_all` takes a replacement step for the suite. Each approach is the same
`outreach_angle` call with a different system prompt, so the only thing that changes is
what goes in the context.

In [ ]:
suite_cases = [json.loads(l) for l in (ROOT / "evals" / "m07_outreach_angle.jsonl").read_text().splitlines() if l.strip()]
# The JSON schema travels with every request (response_format), and it's billed as input too.
SCHEMA_TOKENS = tokens(json.dumps(outreach.SCHEMA))

def step_for(build):
    def step(inp):
        return outreach.outreach_angle(inp["company"], version="v3", system=build(inp["company"]))
    return step

runs = {}
for name, build in APPROACHES.items():
    t0 = time.perf_counter()
    report = run_all(only=["m07_outreach_angle"], steps={"m07_outreach_angle": step_for(build)}, quiet=True)
    s = report["suites"]["m07_outreach_angle"]
    in_tokens = [tokens(build(c["input"]["company"])) + SCHEMA_TOKENS
                 + tokens(outreach.fence(c["input"]["company"]["description"]) + "\nReply with the JSON object only.")
                 for c in suite_cases]
    runs[name] = {
        "pass_rate": s["pass_rate"],
        "input_tokens_per_lead": sum(in_tokens) / len(in_tokens),
        "prefix_tokens": tokens(build(suite_cases[0]["input"]["company"])) + SCHEMA_TOKENS if name != "C: + retrieved facts" else None,
        "seconds_per_lead": (time.perf_counter() - t0) / s["total"],
    }
    print(f"{name:<24} pass {s['passed']}/{s['total']}  input tokens/lead {runs[name]['input_tokens_per_lead']:.0f}")
print(f"(each includes {SCHEMA_TOKENS} tokens of JSON schema)")

## 4. Output tokens

Output length is about the same for every approach: one JSON object. Estimate it from the
worked example's reply in the prompt, which is the length you asked for. With a key, read
the real number from `logs/llm_calls.jsonl` instead.

In [ ]:
example_reply = SYSTEM_B.split("Reply:\n", 1)[1]
OUTPUT_TOKENS = tokens(example_reply)
print("estimated output tokens per lead:", OUTPUT_TOKENS)

## 5. YOUR TURN: your prices

Prices change and differ by model, so they're inputs, not facts. Copy them from your
provider's pricing page for the model you'd use. Until you fill them in, the cost
columns stay empty.

In [ ]:
PRICES = {
    "input_per_million": None,    # USD per million input tokens, from your provider's pricing page
    "output_per_million": None,   # USD per million output tokens
}
LEADS_PER_MONTH = 2_000           # example input: how many leads this step runs on

def cost_per_lead(input_tokens, output_tokens=OUTPUT_TOKENS, prices=PRICES):
    if None in prices.values():
        return float("nan")  # no prices yet
    return (input_tokens * prices["input_per_million"] + output_tokens * prices["output_per_million"]) / 1_000_000

table = pd.DataFrame({
    name: {
        "suite pass rate" + (" (MOCK)" if mock_mode() else ""): r["pass_rate"],
        "input tokens / lead": round(r["input_tokens_per_lead"]),
        "cost / lead (USD)": cost_per_lead(r["input_tokens_per_lead"]),
        "cost / month (USD)": cost_per_lead(r["input_tokens_per_lead"]) * LEADS_PER_MONTH,
        "seconds / lead" + (" (MOCK)" if mock_mode() else ""): r["seconds_per_lead"],
    } for name, r in runs.items()
}).T
table

## 6. Does caching change anything here?

Caching discounts a repeated prefix, but only above a minimum length. Module 7 found
minimums of 1,024 tokens on OpenAI's current models and 512 to 4,096 by model on
Anthropic's (checked November 11, 2026). Compare the minimum with your measured prefix.

In [ ]:
CACHE_MIN_TOKENS = 1024          # example input: your model's minimum cacheable prefix
prefix_b = runs["B: prompt + 1 example"]["prefix_tokens"]
print(f"B's fixed prefix (system prompt + schema) is {prefix_b} tokens; the minimum is {CACHE_MIN_TOKENS}.")
print("caching applies" if prefix_b >= CACHE_MIN_TOKENS else "too short to cache: caching saves nothing on this step")

## 7. Fine-tuning break-even

The honest case for fine-tuning B: a tuned model learns the format and style, so you can
drop the worked example and send A's shorter prompt. The saving per lead is the example's
tokens. Against that: a one-off training cost, and a higher price per token for the tuned
model on most platforms. All three are inputs.

In [ ]:
# YOUR TURN: replace these with numbers from a platform that still offers fine-tuning
# (see the module page for what's available).
FT = {
    "training_cost": 50.00,         # one-off USD for a training run
    "price_multiplier": 1.5,        # tuned model's price per token vs the base model's
}
plain_b = cost_per_lead(runs["B: prompt + 1 example"]["input_tokens_per_lead"])
tuned_prices = {k: None if v is None else v * FT["price_multiplier"] for k, v in PRICES.items()}
tuned_a = cost_per_lead(runs["A: prompt only"]["input_tokens_per_lead"], prices=tuned_prices)
saving = plain_b - tuned_a
print(f"B on the base model: ${plain_b:.6f} per lead")
print(f"A on a tuned model:  ${tuned_a:.6f} per lead")
if saving != saving:  # NaN: prices not entered yet
    print("enter your prices in section 5 first")
elif saving > 0:
    print(f"break-even after {FT['training_cost'] / saving:,.0f} leads")
else:
    print(f"never breaks even: the tuned model costs ${-saving:.6f} MORE per lead, before the training cost")

# Sensitivity: the best case, a tuned model at the base model's price.
best_saving = plain_b - cost_per_lead(runs["A: prompt only"]["input_tokens_per_lead"])
if best_saving == best_saving:
    print(f"best case (same price): saves ${best_saving:.6f} per lead, "
      f"break-even after {FT['training_cost'] / best_saving:,.0f} leads "
      f"= {FT['training_cost'] / best_saving / LEADS_PER_MONTH:,.0f} months at {LEADS_PER_MONTH:,} leads a month")

## 8. The memo

In [ ]:
# YOUR TURN: read the table, the caching check and the break-even, then write the memo.
memo = {
    "compared": list(runs),
    "suite": "m07_outreach_angle (20 cases)",
    "recommendation": "",          # which approach, and why, in a few sentences
    "what_would_change_it": [],    # at least two things that would change your answer
}
print(json.dumps(memo, indent=2))

## Done-when checks

In [ ]:
from lib.checks import done_when

_ = done_when([
    ("At least two approaches ran on the same eval suite", len(runs) >= 2 and all(r["pass_rate"] is not None for r in runs.values())),
    ("Cost per lead comes from measured tokens times prices you entered",
     all(v is not None for v in PRICES.values()) and table["cost / lead (USD)"].gt(0).all()),
    ("The memo makes a recommendation", len(memo["recommendation"].split()) >= 20),
    ("The memo says what would change the decision", len(memo["what_would_change_it"]) >= 2),
])